# DT2 – granskad beräkning
Granskningskopia av `constraintanalysis.ipynb`. Originalet är oförändrat.

**Bekräfta först:** `CD0_reference_area_m2` från VSP. Den sparade originalcellen använder 930 m² för vågmotstånd men ungefär 530 m² för cruise-CL. Om 930 är en ny fysisk vingarea måste även modellens massa, AR och geometriuppgifter uppdateras.

Alla koefficienter nedan normaliseras till `S_model`. Geometrin hålls fast medan cruise-CL varierar. Ett constraintdiagram är därför en lokal dimensioneringsuppskattning; efter val av ny vingarea behövs en geometriuppdatering.

Kvarvarande antaganden: e=0.8; samma preliminära CLmax för start och landning; climb-CD0 approximeras med cruise-CD0; inga explicit modellerade klaff-/landställstillägg i climb. TOP=239 för två motorer är användarens diagramavläsning och inte självständigt verifierad.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator, CubicHermiteSpline

g = 9.82
rho_sl = 1.225
m0 = 295985.59
S_model = 530.36  # Senast bekräftad fysisk referensarea [m²]
AR = 10.0        # Antagande: kontrollera b²/S mot aktuell modell
e = 0.8

CD0_reported = 0.01407
CD0_reference_area_m2 = None  # FYLL I faktisk referensarea från VSP, inte ett gissat värde

f_landing = 0.8513
f_cruise_start = 0.9813
f_cruise_end = 0.8560
f_climb = 0.9813
CLmax_landing = 2.4  # Behåll aktuellt antagande; prova t.ex. 2.3 separat
CLmax_takeoff = 2.4  # Separat antagande; måste motiveras för startklaff
CL_TO = CLmax_takeoff / 1.21
TOP_Raymer = 239.0
TOP = TOP_Raymer * 0.45359237 / 0.09290304
v_app_knots = 146.0
v_stall = v_app_knots / 1.23 * 0.5144
LFL = 2300.0
M_cruise = 0.85
M_climb = 0.4
V_cruise = M_cruise * 296.5
V_climb = M_climb * 338.55
q_cruise = 0.5 * 0.3796 * V_cruise**2
q_climb = 0.5 * 1.1721 * V_climb**2
ROC_required = 2800 * 0.3048 / 60  # m/s
# Uppfyll både angivet ROC- och 6-graderskrav vid samma fart.
gamma_climb = np.arcsin(max(ROC_required / V_climb, np.sin(np.deg2rad(6))))

A_max = 93.5  # m²; kontrollera yttre tvärsnittsareakurva
l_effective = 82.53 - (52 - 13)  # m; kontrollera konstanta intervall i TOTAL area
sweep_LE = 25.0
Ewd = 1.8
MddL0 = 0.82  # Originalets aktuella värde; markeringen i bifogad graf var ca 0.78
Mddfus = 0.77
LF_at_CL_end = 0.93  # Originalets värde; grafmarkeringen var ca 0.91
# Ingen LF(CL)-kurva har digitaliserats. Baslinjen fryser LF vid referensvärdet.
# Detta begränsar giltigheten vid stora ändringar av CL/vingbelastning.
WS_model = m0 / S_model
CL_start = f_cruise_start * WS_model * g / q_cruise
CL_end = f_cruise_end * WS_model * g / q_cruise
print(f"W/S start = {WS_model:.3f} kg/m²; CL cruise start/end = {CL_start:.4f}/{CL_end:.4f}")


In [ ]:
def thrust_lapse(M, T, p, TR=1.0):
    theta0 = T / 288.15 * (1 + 0.2 * M**2)
    delta0 = p / 101325 * (1 + 0.2 * M**2)**3.5
    correction = 0 if theta0 <= TR else 3 * (theta0 - TR) / (1.5 + M)
    return delta0 * (1 - 0.49 * np.sqrt(M) - correction)

lapse_cruise = thrust_lapse(M_cruise, 218.81, 23842)
lapse_climb = thrust_lapse(M_climb, 285.68, 95952.5)

# Båda vertikala gränser uttrycks vid STARTMASSAN.
WS_approach_TO = (0.5 * rho_sl * v_stall**2 / g * CLmax_landing) / f_landing
WS_landing_TO = ((LFL / 1.67 - 305) * CLmax_landing / 4.84) / f_landing
WS_limit = min(WS_approach_TO, WS_landing_TO)
CL_required_approach = f_landing * WS_model * g / (0.5 * rho_sl * v_stall**2)
CL_required_landing = 4.84 * f_landing * WS_model / (LFL / 1.67 - 305)
print(f"Approachgräns = {WS_approach_TO:.2f}; landningsgräns = {WS_landing_TO:.2f} kg/m² vid start")
print(f"Erforderligt CLmax: approach {CL_required_approach:.4f}, landing {CL_required_landing:.4f}")
print(f"Climb-vinkel som uppfyller båda fartbundna kraven: {np.rad2deg(gamma_climb):.3f} deg")


## Dragpolar – kräver bekräftad VSP-referensarea
Om CD0=0.01407 redan avser 530.36 m², fyll i 530.36. Om samma geometri analyserades med normaliseringsarean 930 m², fyll i 930: CD0 skalas då till rätt referensarea. Om geometrin också ändrades är en ren omskalning otillräcklig.

Vågmotståndet beräknas med samma Hermite/PCHIP-metod som i originalet, med Mcrit-gräns och kontroll av ordningen på interpolationspunkterna. Ingen omräkning `L/D / 0.866` görs: cruise-L/D är inte automatiskt L/D vid maximal räckvidd.


In [ ]:
if CD0_reference_area_m2 is None:
    raise ValueError("Bekräfta CD0_reference_area_m2 från VSP i indatacellen och kör om.")
assert CD0_reference_area_m2 > 0 and S_model > 0 and l_effective > 0
C_D0 = CD0_reported * CD0_reference_area_m2 / S_model
K_induced = 1 / (np.pi * AR * e)
K_viscous = 0.38 * C_D0

def wave_drag_curve(CL, base_Mdd=MddL0, LF=LF_at_CL_end):
    Mdd = min(base_Mdd * LF - 0.05 * CL, Mddfus)
    Mcrit = Mdd - 0.08
    M_super = np.linspace(1.2, 2.0, 20)
    dq_sh = 9 * np.pi / 2 * (A_max / l_effective)**2
    cd_super = Ewd * (1 - 0.386 * (M_super - 1.2)**0.57 *
                      (1 - np.pi * sweep_LE**0.77 / 100)) * dq_sh / S_model
    x = np.r_[[Mcrit, Mdd, 1.0, 1.05], M_super]
    y = np.r_[[0, 0.002, cd_super[0] / 2, cd_super[0]], cd_super]
    if not np.all(np.diff(x) > 0):
        raise ValueError("Machtalspunkterna är inte strikt stigande; denna interpolationsmall gäller inte.")
    slopes = PchipInterpolator(x, y).derivative()(x)
    slopes[3] = 0.5 * (y[3] - y[2]) / (x[3] - x[2])
    return CubicHermiteSpline(x, y, slopes, extrapolate=False), Mcrit, Mdd

def wave_cd(CL, Mach=M_cruise, base_Mdd=MddL0, LF=LF_at_CL_end):
    curve, Mcrit, _ = wave_drag_curve(CL, base_Mdd, LF)
    if Mach <= Mcrit:
        return 0.0
    value = float(curve(Mach))
    if not np.isfinite(value) or value < 0:
        raise ValueError("Ogiltigt vågmotstånd: kontrollera interpolationsdomän och indata.")
    return value

def cruise_drag(CL):
    return C_D0 + (K_induced + K_viscous) * CL**2 + wave_cd(CL)

for name, CL in [('Cruise start', CL_start), ('Cruise end', CL_end)]:
    CDw = wave_cd(CL)
    CD = cruise_drag(CL)
    print(f"{name}: CL={CL:.4f}, CD0={C_D0:.5f}, CDi={K_induced*CL**2:.5f}, "
          f"CDvisc={K_viscous*CL**2:.5f}, CDwave={CDw:.5f}, L/D={CL/CD:.3f}")

# Känslighetsfall, inte en godkänd ändring av indata:
CDw_marked = wave_cd(CL_end, base_Mdd=0.78, LF=0.91)
print(f"Bifogade grafers ungefärliga markeringar (0.78, 0.91): CDwave={CDw_marked:.5f}")
curve, Mcrit, Mdd = wave_drag_curve(CL_end)
x = np.linspace(Mcrit, 2, 250)
fig, ax = plt.subplots()
ax.plot(x, curve(x), label='Wave drag, cruise-end CL')
ax.scatter([M_cruise], [wave_cd(CL_end)], color='red', label='Cruise M=0.85')
ax.set(xlabel='Mach', ylabel='CD wave')
ax.grid(); ax.legend(); plt.show()


## Uppdaterade constraints
För planflykt: `CL = f * WS_TO * g / q` och `TW_TO = f / thrust_lapse * CD / CL`.
För climb används `CL = f * WS_TO * g * cos(gamma) / q` och `TW_TO = f / thrust_lapse * (CD/CL*cos(gamma) + sin(gamma))`.

Cruise visas vid både början och slutet av segmentet. LF är tills vidare konstant; uppdatera LF från Raymer om en annan designpunkt väljs. Kurvorna är en lokal uppskattning med nuvarande geometri, inte en färdig optimering av vingen.


In [ ]:
def tw_cruise(ws, fraction):
    CL = fraction * ws * g / q_cruise
    return fraction / lapse_cruise * cruise_drag(CL) / CL

def tw_climb(ws):
    CL = f_climb * ws * g * np.cos(gamma_climb) / q_climb
    CD = C_D0 + (K_induced + K_viscous) * CL**2
    return f_climb / lapse_climb * (CD / CL * np.cos(gamma_climb) + np.sin(gamma_climb))

def tw_takeoff(ws):
    return ws / (TOP * CL_TO)

ws = np.linspace(300, 750, 451)
to = tw_takeoff(ws)
climb = tw_climb(ws)
crs_start = np.array([tw_cruise(w, f_cruise_start) for w in ws])
crs_end = np.array([tw_cruise(w, f_cruise_end) for w in ws])
envelope = np.maximum.reduce([to, climb, crs_start, crs_end])
current = {'Takeoff': tw_takeoff(WS_model), 'Climb': tw_climb(WS_model),
           'Cruise start': tw_cruise(WS_model, f_cruise_start),
           'Cruise end': tw_cruise(WS_model, f_cruise_end)}
for name, value in current.items(): print(f"{name}: required T_SL/W_TO = {value:.4f}")
required = max(current.values())
print(f"Nuvarande area: erforderlig total havsnivådragkraft = {required*m0*g/1000:.1f} kN")
print(f"Approach/landing tillåter nuvarande area: {WS_model <= WS_limit}")
fig, ax = plt.subplots(figsize=(10, 6))
for values, label in [(to,'Takeoff'),(climb,'Climb'),(crs_start,'Cruise start'),(crs_end,'Cruise end')]:
    ax.plot(ws, values, label=label)
ax.plot(ws, envelope, color='black', linestyle=':', label='Required envelope')
ax.axvline(WS_approach_TO, color='red', linestyle='--', label='Approach limit, TO basis')
ax.axvline(WS_landing_TO, color='orange', linestyle='--', label='Landing limit, TO basis')
ax.scatter([WS_model], [required], color='black', label='Current wing: required thrust')
ax.set(xlabel='m_TO / S [kg/m²]', ylabel='T_SL / W_TO [-]', xlim=(300,750))
ax.grid(); ax.legend(); plt.show()


## Slutsatser från kodgranskningen
- Den gamla 530.36 m²-arean kom från en approachgräns utan omvandling till startvikt. Arean är inte därmed ogiltig, men den markerade skärningspunkten var felbenämnd.
- Landningsformlerna `(S_landing-305)*CLmax/4.84` och `(S_landing-305)/(4.84*(1/CLmax))` är identiska.
- Originalets andra cruise-kurva saknade både vågmotstånd och visköst lyftberoende bidrag trots att dessa fanns i L/D-beräkningen.
- Originalets `TOP_Raymer=239` skrevs över med 280; `TOP` räknades inte om i andra iterationen.
- Originalets första dragpolarterm använde gamla `C_D0=0.01175`, medan nästa använde `CD0=0.01407`.
- De senaste sparade grafvärdena 0.82 och 0.93 skiljer sig från de inskickade markeringarna cirka 0.78 och 0.91. Bekräfta profil och avläsning innan ett värde väljs.
- Inget nytt MTOW-resultat har beräknats här. Mata först ett verifierat representativt L/D till missionens viktberäkning och uppdatera segmentfraktionerna därifrån.
- Beräkningarna ovan förutsätter att den rapporterade nya startmassan gäller; calculations.ipynb som tidigare lästes innehöll fortfarande det äldre viktresultatet.
